# Evaluation 02 — PanDerm Base **official encoder + frozen linear probe**

Uses genuine author-supplied PanDerm source and pretrained weights with the saved `linear_probe.joblib` and `model_details.json` from the training notebook. This is **not** a PanDerm full fine-tune. Loading joblib requires trusting the checkpoint producer. Select Kaggle GPU and attach the original source/weights.

**Saved evidence:** `notebook_console.log` captures execution/tracebacks, `run_log.jsonl` records start and completed artifact SHA-256 checksums, and each analysis writes its reported CSV/JSON or graphic files into the configured Kaggle output directory. Keep per-image records and raw console text private; export only reviewed aggregate reports to GitHub.


In [ ]:
# Common Kaggle bootstrap: either attach DermSynReal source as Kaggle Input,
# or set CLONE_REPO_IF_MISSING=True and enable Kaggle Internet.
from pathlib import Path
import subprocess,sys,os,json
REPO_ROOT=Path("/kaggle/input/EDIT_REPO_DATASET/DermSynReal")
CLONE_REPO_IF_MISSING=True
EXPECTED_GIT_SHA=""  # Recommended: set a full pinned SHA for reproducible final runs.
if not (REPO_ROOT/"src/dermsynreal/evaluation.py").is_file():
    if not CLONE_REPO_IF_MISSING:
        raise FileNotFoundError("Attach the repository and change REPO_ROOT, or enable Git clone.")
    REPO_ROOT=Path("/kaggle/working/DermSynReal")
    if not (REPO_ROOT/".git").is_dir():
        subprocess.check_call(["git","clone","https://github.com/shanmukhdatta/DermSynReal.git",str(REPO_ROOT)])
if EXPECTED_GIT_SHA:
    if not (REPO_ROOT/".git").is_dir():
        raise ValueError("For pinned SHA verification attach a Git checkout or enable clone")
    actual=subprocess.check_output(["git","-C",str(REPO_ROOT),"rev-parse","HEAD"],text=True).strip()
    if actual!=EXPECTED_GIT_SHA:
        raise ValueError(f"Repository SHA {actual} differs from pinned SHA {EXPECTED_GIT_SHA}")
elif (REPO_ROOT/".git").is_dir():
    print("Current Git SHA:",subprocess.check_output(["git","-C",str(REPO_ROOT),"rev-parse","HEAD"],text=True).strip())
    print("WARNING: pin EXPECTED_GIT_SHA for final research outputs")
sys.path.insert(0,str(REPO_ROOT/"src"))
print("DermSynReal source:",REPO_ROOT)


In [ ]:
# Install only if packages are absent or incompatible; match the original PanDerm training notebook.
INSTALL_PANDERM_PACKAGES=False
if INSTALL_PANDERM_PACKAGES:
    subprocess.check_call([sys.executable,"-m","pip","install","-q",
                           "timm==0.9.16","open_clip_torch","scikit-learn>=1.4,<2",
                           "pillow","einops","joblib"])
OFFICIAL_PANDERM_REPO="/kaggle/input/EDIT_PANDERM_SOURCE/PanDerm"
OFFICIAL_WEIGHTS="/kaggle/input/EDIT_PANDERM_WEIGHTS/panderm_bb_data6_checkpoint-499.pth"
SAVED_LINEAR_PROBE="/kaggle/input/EDIT_PANDERM_TRAINED/linear_probe.joblib"
MODEL_DETAILS="/kaggle/input/EDIT_PANDERM_TRAINED/model_details.json"
REAL_TEST_MANIFEST="/kaggle/input/EDIT_REAL_SPLIT/real_test_manifest.csv"
REAL_IMAGE_ROOT="/kaggle/input/EDIT_REAL_IMAGES"
SYNTH_TEST_MANIFEST=""
SYNTH_IMAGE_ROOT=""
TRAIN_SYNTHETIC_MANIFEST=""
OUTPUT_CSV="/kaggle/working/dermsynreal_eval_02/panderm_final_predictions.csv"
RUN_FINAL_TEST=False

In [ ]:
# Preserve a complete Kaggle notebook console and structured start event.
# This log can contain image file paths or case notes; keep it out of public GitHub.
from dermsynreal.kaggle_runs import start_notebook_logging,append_run_log
EVALUATION_OUTPUT=Path(Path(OUTPUT_CSV).parent)
PRIVATE_EVALUATION_LOG=start_notebook_logging(EVALUATION_OUTPUT,
    kind='evaluation_02',config={
        'notebook':'02_PanDerm_Official_Probe_Inference.ipynb','data_verified':False,
        'run_enabled':bool(RUN_FINAL_TEST)},repo_root=REPO_ROOT)
print('Saved notebook console:',PRIVATE_EVALUATION_LOG)


In [ ]:
from dermsynreal.panderm_inference import run_panderm_inference
if not RUN_FINAL_TEST:
    print("Test remains locked. Supply trusted PanDerm artifacts, fix paths and enable final test deliberately.")
else:
    import torch
    assert torch.cuda.is_available(),"Select Kaggle GPU accelerator"
    info=run_panderm_inference(official_repo=OFFICIAL_PANDERM_REPO,
      official_weights=OFFICIAL_WEIGHTS,saved_probe=SAVED_LINEAR_PROBE,
      model_details=MODEL_DETAILS,real_manifest=REAL_TEST_MANIFEST,
      real_image_root=REAL_IMAGE_ROOT,synthetic_manifest=SYNTH_TEST_MANIFEST or None,
      synthetic_image_root=SYNTH_IMAGE_ROOT or None,
      train_synthetic_manifest=TRAIN_SYNTHETIC_MANIFEST or None,
      out_csv=OUTPUT_CSV,allow_test=True)
    print(json.dumps(info,indent=2))
    from dermsynreal.evaluation import read_predictions,source_metrics
    print(json.dumps({s:{k:v for k,v in m.items() if k in ("macro_f1","accuracy","n")}
         for s,m in source_metrics(read_predictions(OUTPUT_CSV)).items()},indent=2))


If the official source release changes, stop and reconcile the `get_encoder` API with your training notebook. Do not load a generic ViT substitute.

In [ ]:
# Record checksums only for files actually created by this executed run.
if RUN_FINAL_TEST:
    _files=[Path(OUTPUT_CSV),Path(OUTPUT_CSV).with_suffix('.provenance.json')]
    _log=append_run_log(EVALUATION_OUTPUT,kind='evaluation_02_finished',
        config={'notebook':'02_PanDerm_Official_Probe_Inference.ipynb','status':'completed_artifacts_written'},
        artifacts=[p for p in _files if p.is_file()],repo_root=REPO_ROOT)
    print('Evidence hashes:',_log['artifacts_sha256'])
